# Fine-tune CLIP on Title-Thumbnail Pairs

Code authored by: Shaw Talebi

[Video link](https://youtu.be/W4s6b2ZM6kI) | [Blog link](https://medium.com/towards-data-science/fine-tuning-multimodal-embedding-models-bf007b1c5da5) <br>
[Dataset](https://huggingface.co/datasets/shawhin/yt-title-thumbnail-pairs) | [Fine-tuned Model](https://huggingface.co/shawhin/clip-title-thumbnail-embeddings)

### imports

In [3]:
from datasets import load_dataset
from huggingface_hub import upload_folder

from PIL import Image
import requests

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.evaluation import TripletEvaluator, SentenceEvaluator

from typing import List, Dict
import torch

### import model and dataset

In [1]:
resume_training_from_hf_model = True
finetuned_model_name = "clip-title-thumbnail-embeddings"
finetuned_model_output_dir = f"models/{finetuned_model_name}"
finetuned_model_hf_path = f"pkqinys/shawhin-{finetuned_model_name}"

In [4]:
if resume_training_from_hf_model:
    model = SentenceTransformer(finetuned_model_hf_path)
else:
    model_name = "sentence-transformers/clip-ViT-L-14"
    model = SentenceTransformer(model_name)

README.md: 0.00B [00:00, ?B/s]

In [5]:
dataset = load_dataset("pkqinys/shawhin-yt-title-thumbnail-pairs")

### freeze model params

In [6]:
# pick specific layers to train (note: you can add more layers to this list)
trainable_layers_list = ['projection']

# Apply freezing configuration
for name, param in model.named_parameters():
    # freeze all params
    param.requires_grad = False

    # unfreeze layers in trainable_layers_list
    if any(layer in name for layer in trainable_layers_list):
        param.requires_grad = True

In [7]:
# Verify trainable parameters
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Trainable: {name}")

Trainable: 0.model.visual_projection.weight
Trainable: 0.model.text_projection.weight


In [8]:
# Count total and trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Percentage of trainable parameters: {100 * trainable_params / total_params:.2f}%")

Total parameters: 427,616,513
Trainable parameters: 1,376,256
Percentage of trainable parameters: 0.32%


### preprocess data

In [9]:
# process positive pairs
def preprocess(batch):
    """
        Preprocessing data without augmentations for test set
    """
    # get images from urls
    image_list = [Image.open(requests.get(url, stream=True).raw) for url in batch["thumbnail_url"]]

    # return columns with standard names
    return {
        "anchor": image_list,       
        "positive": batch["title"],  
        "negative": batch["title_neg"]
    }

In [10]:
# remove columns not relevant to training
columns_to_remove = [col for col in dataset['train'].column_names if col not in ['anchor', 'positive', 'negative']]
# applu transformations
dataset = dataset.map(preprocess, batched=True, remove_columns=columns_to_remove)

In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 75
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 16
    })
    test: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 17
    })
})

In [12]:
df = dataset.data['train'].to_pandas()
df

,anchor,positive,negative
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,A Practical Introduction to Large Language Mod...,Prompt Engineering: How to Trick AI into Solvi...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,How to Build a Notion AI Agent (in 18 minutes),4 Ways to Measure Fat Tails with Python (+ Exa...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Context Engineering Explained (5 Practical Tips),"Why I Quit My $150,000 Data Science Job"
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,10 Decision Trees are Better Than 1 | Random F...,LLM Workflows: From Automation to AI Agents (w...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,"3 Ways to Make a Custom AI Assistant | RAG, To...",Detecting Power Laws in Real-world Data | w/ P...
...,...,...,...
70,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,What Nature Can Teach Us About Business...,Multimodal RAG: A Beginner-friendly Guide (wit...
71,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Prompt Engineering: How to Trick AI into Solvi...,Dimensionality Reduction & Segmentation with D...
72,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Local LLM Fine-tuning on Mac (M1 16GB),How to Communicate Effectively (as a Data Scie...
73,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,How to Be Antifragile | 7 Practical Tips,Python QuickStart for People Learning AI [Mini...


### eval pre-trained model

In [13]:
def create_triplet_evaluator(set_name):
    """
        Create triplet evaluator for "train", "valid", or "test" split
    """

    return TripletEvaluator(
        anchors=dataset[f"{set_name}"]["anchor"],
        positives=dataset[f"{set_name}"]["positive"],
        negatives=dataset[f"{set_name}"]["negative"],
        name=f"yt-title-thumbnail-{set_name}",
    )

In [14]:
evaluator_train = create_triplet_evaluator("train")
evaluator_valid = create_triplet_evaluator("valid")

print("Train:", evaluator_train(model))
print("Valid:", evaluator_valid(model))

Train: {'yt-title-thumbnail-train_cosine_accuracy': np.float64(1.0)}
Valid: {'yt-title-thumbnail-valid_cosine_accuracy': np.float64(0.9375)}


In [15]:
class ImageTextRetrievalEvaluator(SentenceEvaluator):
    def __init__(
        self,
        images: List,
        texts: List[str],
        name: str = '',
        k: int = 1,
        batch_size: int = 32,
        show_progress_bar: bool = False
    ):
        self.images = images
        self.texts = texts
        self.name = name
        self.k = k
        self.batch_size = batch_size
        self.show_progress_bar = show_progress_bar

    def __call__(self,
        model: SentenceTransformer,
        output_path: str = None,
        epoch: int = -1,
        steps: int = -1) -> Dict[str, float]:
        
        # Get embeddings for all images
        img_embeddings = model.encode(
            self.images,
            batch_size=self.batch_size,
            show_progress_bar=self.show_progress_bar,
            convert_to_tensor=True
        )
        
        # Get embeddings for all texts
        text_embeddings = model.encode(
            self.texts,
            batch_size=self.batch_size,
            show_progress_bar=self.show_progress_bar,
            convert_to_tensor=True
        )
        
        # Compute similarity matrix
        cos_scores = torch.nn.functional.cosine_similarity(
            img_embeddings.unsqueeze(1),
            text_embeddings.unsqueeze(0),
            dim=2
        )
        
        # Get indices of top k predictions for each image
        _, top_indices = torch.topk(cos_scores, k=self.k, dim=1)
        
        # Calculate Recall@k (correct if ground truth index is in top k predictions)
        correct = sum(i in top_indices[i].tolist() for i in range(len(self.images)))
        recall_at_k = correct / len(self.images)

        return {f'{self.name}_Recall@{self.k}': recall_at_k}

In [16]:
def create_recall_evaluator(set_name, k=1):
    """
        Create triplet evaluator for "train", "valid", or "test" split
    """

    return ImageTextRetrievalEvaluator(
        images=dataset[f"{set_name}"]["anchor"],
        texts=dataset[f"{set_name}"]["positive"],
        name=f"yt-title-thumbnail-{set_name}",
        k=k
    )

In [17]:
# Create new evaluator with Recall@k
evaluator_recall_train = create_recall_evaluator("train", k=1)
evaluator_recall_valid = create_recall_evaluator("valid", k=1)

print("Train:", evaluator_recall_train(model))
print("Valid:", evaluator_recall_valid(model))

Train: {'yt-title-thumbnail-train_Recall@1': 0.8933333333333333}
Valid: {'yt-title-thumbnail-valid_Recall@1': 0.75}


### define training args

In [29]:
# define loss (note: loss expects columns to be ordered as anchor-positive-negative)
loss = MultipleNegativesRankingLoss(model)

# hyperparameters
num_epochs = 4
batch_size = 4
lr = 1e-5

train_args = SentenceTransformerTrainingArguments(
    output_dir=finetuned_model_output_dir,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=lr,
    # Evaluation settings
    eval_strategy="epoch",
    eval_steps=1,
    logging_steps=1,
)

### fine-tune model

In [30]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=train_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["valid"],
    loss=loss,
    evaluator=[evaluator_recall_train, evaluator_recall_valid],
)

In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,Yt-title-thumbnail-train Recall@1,Yt-title-thumbnail-valid Recall@1,Sequential Score
1,0.245600,0.782348,0.880000,0.750000,0.750000
2,0.067000,0.783679,0.906667,0.750000,0.750000
3,0.104900,0.781113,0.946667,0.750000,0.750000
4,0.223400,0.781225,0.946667,0.750000,0.750000


TrainOutput(global_step=76, training_loss=0.17591007786655896, metrics={'train_runtime': 293.5954, 'train_samples_per_second': 1.022, 'train_steps_per_second': 0.259, 'total_flos': 0.0, 'train_loss': 0.17591007786655896, 'epoch': 4.0})

### evaluate fine-tuned model

In [23]:
evaluator_test = create_triplet_evaluator("test")

print("Train:", evaluator_train(model))
print("Valid:", evaluator_valid(model))
print("Test:", evaluator_valid(model))

Train: {'yt-title-thumbnail-train_cosine_accuracy': np.float64(1.0)}
Valid: {'yt-title-thumbnail-valid_cosine_accuracy': np.float64(0.9375)}
Test: {'yt-title-thumbnail-valid_cosine_accuracy': np.float64(0.9375)}


In [32]:
evaluator_recall_test = create_recall_evaluator("test")

print("Train:", evaluator_recall_train(model))
print("Valid:", evaluator_recall_valid(model))
print("Test:", evaluator_recall_test(model))

Train: {'yt-title-thumbnail-train_Recall@1': 0.9466666666666667}
Valid: {'yt-title-thumbnail-valid_Recall@1': 0.75}
Test: {'yt-title-thumbnail-test_Recall@1': 0.8235294117647058}


### push model to hub

In [33]:
# model.push_to_hub(finetuned_model_hf_path)

checkpoint = 'checkpoint-76'
upload_folder(
    folder_path=f"{finetuned_model_output_dir}/{checkpoint}",  # e.g. folder with pytorch_model.bin, config.json
    repo_id=finetuned_model_hf_path,
    commit_message="Update: 4_4_5"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/pkqinys/shawhin-clip-title-thumbnail-embeddings/commit/2d03194905d092364f1d579dca31150cc353255c', commit_message='Update: 4_4_5', commit_description='', oid='2d03194905d092364f1d579dca31150cc353255c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pkqinys/shawhin-clip-title-thumbnail-embeddings', endpoint='https://huggingface.co', repo_type='model', repo_id='pkqinys/shawhin-clip-title-thumbnail-embeddings'), pr_revision=None, pr_num=None)